# Notebook 06 — Source-Split Detection Analysis
**Research Question:** Do detection methods flag `kaggle_ai` (known-spurious) text features
at a higher rate than `google_real` (unknown-status) text features?

This directly answers **RQ1 for the text modality** by splitting the dataset by source and
comparing rejection rates. The key metric is **Δ = kaggle_rejection% − google_rejection%**.
A consistently positive Δ means methods discriminate; near-zero means no discrimination.


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import TwoSlopeNorm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# ── Paths — adjust if needed ──────────────────────────────────────────────────
HERE     = os.path.expanduser("~/spurious_detection")
DATA     = os.path.join(HERE, "datasets", "master_final.parquet")
OUTDIR   = os.path.join(HERE, "results", "notebook06")
os.makedirs(OUTDIR, exist_ok=True)

# ── Detection params ──────────────────────────────────────────────────────────
TARGET       = "next_day_return"
ALPHA        = 0.05
N_BOOTSTRAP  = 500
BLOCK_SIZE   = 20
N_JOBS       = 16
TRAIN_WIN    = 30
TEST_WIN     = 10
WF_THRESHOLD = 0.5
EMBED_CAP    = 50   # cap embeddings for bootstrap runtime

# ── Plot style ────────────────────────────────────────────────────────────────
plt.style.use("dark_background")
KAGGLE_COLOR = "#E05C5C"
GOOGLE_COLOR = "#5B8FD4"
DELTA_COLOR  = "#F5C542"
SENT_COLOR   = "#A78BFA"
LDA_COLOR    = "#34D399"
EMBED_COLOR  = "#60A5FA"

print("Config OK")


## 1. Load Data & Define Feature Groups

In [ ]:
df = pd.read_parquet(DATA)
print(f"Total rows: {len(df)}, columns: {len(df.columns)}")
print(f"source_type counts:\n{df['source_type'].value_counts()}")

# ── Feature lists ─────────────────────────────────────────────────────────────
SENTIMENT_FEATS = [c for c in df.columns if c == "sentiment_score"]
LDA_FEATS       = [c for c in df.columns if c.startswith("lda_topic_")]
EMBED_FEATS     = [c for c in df.columns if c.startswith("embed_")]
ALL_TEXT        = SENTIMENT_FEATS + LDA_FEATS + EMBED_FEATS

# Bootstrap uses capped embeddings to keep runtime reasonable
EMBED_BOOT      = [f"embed_{i}" for i in range(EMBED_CAP) if f"embed_{i}" in df.columns]
TEXT_BOOT       = SENTIMENT_FEATS + LDA_FEATS + EMBED_BOOT

print(f"\nText features: {len(ALL_TEXT)} total")
print(f"  Sentiment: {len(SENTIMENT_FEATS)}")
print(f"  LDA:       {len(LDA_FEATS)}")
print(f"  Embeddings:{len(EMBED_FEATS)} (bootstrap capped at {len(EMBED_BOOT)})")


## 2. Split by Source Type

In [ ]:
df_kaggle = df[df["source_type"] == "kaggle_ai"].copy().dropna(subset=[TARGET])
df_google = df[df["source_type"] == "google_real"].copy().dropna(subset=[TARGET])

print(f"kaggle_ai rows:   {len(df_kaggle)}")
print(f"google_real rows: {len(df_google)}")

if len(df_kaggle) < 100 or len(df_google) < 100:
    print("WARNING: One subset has < 100 rows — results may be unreliable")


## 3. Detection Helpers

In [ ]:
def strategy_returns(feature_series, target_series):
    median = feature_series.median()
    signal = np.where(feature_series >= median, 1, -1)
    return signal * target_series.values

def sharpe(returns, annualize=252):
    r = np.array(returns)
    return 0.0 if r.std() == 0 else (r.mean() / r.std()) * np.sqrt(annualize)

def block_bootstrap_indices(n, block_size, rng):
    indices = []
    while len(indices) < n:
        start = rng.integers(0, max(1, n - block_size))
        indices.extend(range(start, start + block_size))
    return indices[:n]

def one_boot_iter(seed, feature_matrix, target_vals, block_size):
    rng = np.random.default_rng(seed)
    idx = block_bootstrap_indices(len(target_vals), block_size, rng)
    t = target_vals[idx]
    return np.array([
        sharpe(strategy_returns(pd.Series(feature_matrix[idx, i]), pd.Series(t)))
        for i in range(feature_matrix.shape[1])
    ])

print("Helpers defined")


## 4. Bonferroni / FDR Detection

In [ ]:
def run_bonferroni_fdr(subset_df, features):
    rows = []
    for feat in features:
        valid = subset_df[[feat, TARGET]].dropna()
        if len(valid) < 30:
            continue
        r, p = stats.pearsonr(valid[feat], valid[TARGET])
        rows.append({"feature": feat, "pearson_r": r, "p_value": p,
                     "abs_r": abs(r)})
    if not rows:
        return pd.DataFrame()
    res = pd.DataFrame(rows)
    K = len(res)
    bonf_thresh = ALPHA / K
    reject_fdr, _, _, _ = multipletests(res["p_value"], alpha=ALPHA, method="fdr_bh")
    res["bonferroni_rejected"] = res["p_value"] < bonf_thresh
    res["fdr_rejected"]        = reject_fdr
    return res

print("Running Bonferroni/FDR on kaggle...")
bonf_kaggle = run_bonferroni_fdr(df_kaggle, ALL_TEXT)
bonf_kaggle["source"] = "kaggle_ai"

print("Running Bonferroni/FDR on google...")
bonf_google = run_bonferroni_fdr(df_google, ALL_TEXT)
bonf_google["source"] = "google_real"

print(f"Kaggle  — Bonferroni rej: {bonf_kaggle['bonferroni_rejected'].mean():.1%}  FDR rej: {bonf_kaggle['fdr_rejected'].mean():.1%}")
print(f"Google  — Bonferroni rej: {bonf_google['bonferroni_rejected'].mean():.1%}  FDR rej: {bonf_google['fdr_rejected'].mean():.1%}")


## 5. Bootstrap Detection
*Embeddings capped at first 50 dims for runtime.*

In [ ]:
def run_bootstrap(subset_df, features, n_boot=N_BOOTSTRAP):
    valid = subset_df[features + [TARGET]].dropna()
    target_vals = valid[TARGET].values
    feat_matrix = valid[features].values
    obs_sharpes = np.array([
        sharpe(strategy_returns(pd.Series(feat_matrix[:, i]), pd.Series(target_vals)))
        for i in range(len(features))
    ])
    boot_dist = np.array(
        Parallel(n_jobs=N_JOBS)(
            delayed(one_boot_iter)(seed, feat_matrix, target_vals, BLOCK_SIZE)
            for seed in range(n_boot)
        )
    )
    critical = np.percentile(boot_dist, 95, axis=0)
    rejected = obs_sharpes > critical
    return pd.DataFrame({
        "feature": features,
        "obs_sharpe": obs_sharpes,
        "critical_95": critical,
        "bootstrap_rejected": rejected
    })

print("Running Bootstrap on kaggle... (this may take a few minutes)")
boot_kaggle = run_bootstrap(df_kaggle, TEXT_BOOT)
boot_kaggle["source"] = "kaggle_ai"

print("Running Bootstrap on google...")
boot_google = run_bootstrap(df_google, TEXT_BOOT)
boot_google["source"] = "google_real"

print(f"Kaggle  — Bootstrap rej: {boot_kaggle['bootstrap_rejected'].mean():.1%}")
print(f"Google  — Bootstrap rej: {boot_google['bootstrap_rejected'].mean():.1%}")


## 6. Walk-Forward Detection

In [ ]:
def run_walkforward(subset_df, features):
    subset_df = subset_df.sort_values("date") if "date" in subset_df.columns else subset_df.copy()
    n = len(subset_df)
    results = {f: [] for f in features}
    step = max(1, TEST_WIN)
    starts = range(0, n - TRAIN_WIN - TEST_WIN, step)
    if len(list(starts)) < 3:
        print("  WARNING: Not enough rows for walk-forward, skipping")
        return pd.DataFrame({
            "feature": features,
            "mean_gen_ratio": [np.nan] * len(features),
            "wf_rejected": [False] * len(features)
        })
    for start in starts:
        train = subset_df.iloc[start: start + TRAIN_WIN]
        test  = subset_df.iloc[start + TRAIN_WIN: start + TRAIN_WIN + TEST_WIN]
        for feat in features:
            tr = train[[feat, TARGET]].dropna()
            te = test[[feat, TARGET]].dropna()
            if len(tr) < 5 or len(te) < 3:
                continue
            is_ = sharpe(strategy_returns(tr[feat], tr[TARGET]))
            oos = sharpe(strategy_returns(te[feat], te[TARGET]))
            ratio = oos / is_ if abs(is_) > 0.01 else np.nan
            results[feat].append(ratio)
    rows = []
    for feat in features:
        ratios = [r for r in results[feat] if not np.isnan(r)]
        mean_r = np.mean(ratios) if ratios else np.nan
        rows.append({
            "feature": feat,
            "mean_gen_ratio": mean_r,
            "wf_rejected": (mean_r < WF_THRESHOLD) if not np.isnan(mean_r) else False
        })
    return pd.DataFrame(rows)

print("Running Walk-Forward on kaggle...")
wf_kaggle = run_walkforward(df_kaggle, ALL_TEXT)
wf_kaggle["source"] = "kaggle_ai"

print("Running Walk-Forward on google...")
wf_google = run_walkforward(df_google, ALL_TEXT)
wf_google["source"] = "google_real"

print(f"Kaggle  — Walk-Fwd rej: {wf_kaggle['wf_rejected'].mean():.1%}")
print(f"Google  — Walk-Fwd rej: {wf_google['wf_rejected'].mean():.1%}")


## 7. Summary Table

In [ ]:
def rej_rate(df, col):
    return df[col].mean() if col in df.columns else np.nan

summary_rows = []
for method, col, k_df, g_df in [
    ("Bonferroni",   "bonferroni_rejected", bonf_kaggle, bonf_google),
    ("FDR",          "fdr_rejected",         bonf_kaggle, bonf_google),
    ("Bootstrap",    "bootstrap_rejected",   boot_kaggle, boot_google),
    ("Walk-Forward", "wf_rejected",          wf_kaggle,   wf_google),
]:
    kr = rej_rate(k_df, col)
    gr = rej_rate(g_df, col)
    delta = kr - gr if not np.isnan(kr) and not np.isnan(gr) else np.nan
    summary_rows.append({"method": method, "kaggle_rej": kr, "google_rej": gr, "delta": delta})

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUTDIR, "source_split_summary.csv"), index=False)

print("\n{:<14} {:>12} {:>12} {:>8}".format("Method", "Kaggle Rej%", "Google Rej%", "Delta"))
print("-" * 50)
for _, row in summary.iterrows():
    print("{:<14} {:>11.1%} {:>11.1%} {:>+7.1%}".format(
        row["method"], row["kaggle_rej"], row["google_rej"], row["delta"]))

# Interpret
avg_delta = summary["delta"].mean()
if avg_delta > 0.05:
    print(f"\n✓ Methods consistently flag kaggle more (avg Δ={avg_delta:+.1%}) — discrimination present")
else:
    print(f"\n✗ Methods do NOT consistently discriminate (avg Δ={avg_delta:+.1%}) — null result for text modality")


## 8. Sub-Modality Breakdown

In [ ]:
def submod_label(feat):
    if feat == "sentiment_score": return "Sentiment"
    if feat.startswith("lda_topic_"): return "LDA Topics"
    return "Embeddings"

def submod_summary(k_df, g_df, col):
    rows = []
    for label, feats in [("Sentiment", SENTIMENT_FEATS), ("LDA Topics", LDA_FEATS),
                          ("Embeddings", EMBED_FEATS)]:
        kf = k_df[k_df["feature"].isin(feats)] if "feature" in k_df.columns else pd.DataFrame()
        gf = g_df[g_df["feature"].isin(feats)] if "feature" in g_df.columns else pd.DataFrame()
        kr = kf[col].mean() if col in kf.columns and len(kf) else np.nan
        gr = gf[col].mean() if col in gf.columns and len(gf) else np.nan
        rows.append({"submod": label, "kaggle": kr, "google": gr,
                     "delta": kr - gr if not np.isnan(kr) and not np.isnan(gr) else np.nan})
    return pd.DataFrame(rows)

bonf_sub = submod_summary(bonf_kaggle, bonf_google, "bonferroni_rejected")
fdr_sub  = submod_summary(bonf_kaggle, bonf_google, "fdr_rejected")
boot_sub = submod_summary(boot_kaggle, boot_google, "bootstrap_rejected")
wf_sub   = submod_summary(wf_kaggle,   wf_google,   "wf_rejected")

# Build delta heatmap matrix
methods = ["Bonferroni", "FDR", "Bootstrap", "Walk-Forward"]
submods = ["Sentiment", "LDA Topics", "Embeddings"]
delta_matrix = np.array([
    bonf_sub["delta"].values,
    fdr_sub["delta"].values,
    boot_sub["delta"].values,
    wf_sub["delta"].values,
])
print("Delta matrix (rows=methods, cols=submodalities):")
print(pd.DataFrame(delta_matrix, index=methods, columns=submods).round(3))


## 9. Plot 1 — Rejection Rate Comparison by Method

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor("#0F0F13")
ax.set_facecolor("#0F0F13")

x = np.arange(len(summary))
w = 0.32
bars_k = ax.bar(x - w/2, summary["kaggle_rej"], w, color=KAGGLE_COLOR,
                alpha=0.9, label="kaggle_ai (spurious)", zorder=3)
bars_g = ax.bar(x + w/2, summary["google_rej"], w, color=GOOGLE_COLOR,
                alpha=0.9, label="google_real (unknown)", zorder=3)

# Annotate values
for bar in bars_k:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.0%}",
            ha="center", va="bottom", fontsize=9, color=KAGGLE_COLOR, fontweight="bold")
for bar in bars_g:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.0%}",
            ha="center", va="bottom", fontsize=9, color=GOOGLE_COLOR, fontweight="bold")

# Delta annotations
for i, row in summary.iterrows():
    ax.text(i, max(row["kaggle_rej"], row["google_rej"]) + 0.06,
            f"Δ={row['delta']:+.0%}", ha="center", fontsize=10,
            color=DELTA_COLOR, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(summary["method"], fontsize=12, color="white")
ax.set_ylabel("Rejection Rate", color="white", fontsize=12)
ax.set_ylim(0, 1.18)
ax.set_title("RQ1 Text Split: Do Detection Methods Flag Kaggle (Spurious)\nMore Than Google (Real)?",
             fontsize=13, color="white", pad=15)
ax.legend(fontsize=10, framealpha=0.2)
ax.tick_params(colors="white")
for spine in ax.spines.values(): spine.set_edgecolor("#333")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(axis="y", alpha=0.15, zorder=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "plot1_rejection_rates.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot1")


## 10. Plot 2 — Detection Delta Heatmap by Sub-Modality × Method

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor("#0F0F13")
ax.set_facecolor("#0F0F13")

# Replace nan with 0 for display
dm = np.nan_to_num(delta_matrix, nan=0.0)
vmax = max(0.05, np.nanmax(np.abs(delta_matrix)))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im = ax.imshow(dm, cmap="RdBu_r", norm=norm, aspect="auto")

ax.set_xticks(range(len(submods)))
ax.set_xticklabels(submods, fontsize=12, color="white")
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods, fontsize=12, color="white")
ax.tick_params(colors="white")

for i in range(len(methods)):
    for j in range(len(submods)):
        val = delta_matrix[i, j]
        txt = f"{val:+.0%}" if not np.isnan(val) else "n/a"
        color = "white" if abs(dm[i, j]) > vmax * 0.5 else "black"
        ax.text(j, i, txt, ha="center", va="center", fontsize=13,
                fontweight="bold", color=color)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label("Δ (Kaggle − Google)", color="white", fontsize=10)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:+.0%}"))

ax.set_title("Detection Delta (Kaggle − Google) by Sub-Modality × Method\n"
             "Red = kaggle flagged more  |  Blue = google flagged more",
             fontsize=12, color="white", pad=12)

# Grid lines
for x in np.arange(-0.5, len(submods), 1):
    ax.axvline(x, color="#333", linewidth=0.8)
for y in np.arange(-0.5, len(methods), 1):
    ax.axhline(y, color="#333", linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "plot2_delta_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot2")


## 11. Plot 3 — P-value Distribution: Kaggle vs Google

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
fig.patch.set_facecolor("#0F0F13")

submod_map = [("Sentiment", SENTIMENT_FEATS),
              ("LDA Topics", LDA_FEATS),
              ("Embeddings", EMBED_FEATS)]

for ax, (label, feats) in zip(axes, submod_map):
    ax.set_facecolor("#0F0F13")
    kp = bonf_kaggle[bonf_kaggle["feature"].isin(feats)]["p_value"].dropna()
    gp = bonf_google[bonf_google["feature"].isin(feats)]["p_value"].dropna()
    bins = np.linspace(0, 1, 21)
    ax.hist(kp, bins=bins, alpha=0.65, color=KAGGLE_COLOR, label="kaggle_ai",
            edgecolor="#0F0F13", linewidth=0.5, density=True)
    ax.hist(gp, bins=bins, alpha=0.65, color=GOOGLE_COLOR, label="google_real",
            edgecolor="#0F0F13", linewidth=0.5, density=True)
    ax.axvline(0.05, color=DELTA_COLOR, linestyle="--", linewidth=1.2,
               label=f"α=0.05")
    ax.set_title(label, fontsize=12, color="white", pad=8)
    ax.set_xlabel("p-value", color="white", fontsize=10)
    ax.set_ylabel("Density", color="white", fontsize=10)
    ax.tick_params(colors="white")
    ax.legend(fontsize=8, framealpha=0.15)
    for spine in ax.spines.values(): spine.set_edgecolor("#333")
    ax.grid(alpha=0.1)

fig.suptitle("P-value Distributions: Kaggle vs Google Text Features (Bonferroni)
"
             "Uniform = null true  |  Mass near 0 = genuine signal",
             fontsize=12, color="white", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "plot3_pvalue_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot3")


## 12. Plot 4 — Effect Size Scatter: Kaggle vs Google

In [ ]:
# Merge kaggle/google |r| on feature
def make_r_df(bonf_df, suffix):
    return bonf_df[["feature", "abs_r"]].rename(columns={"abs_r": f"abs_r_{suffix}"})

merged = make_r_df(bonf_kaggle, "kaggle").merge(
         make_r_df(bonf_google, "google"), on="feature", how="inner")
merged["submod"] = merged["feature"].map(submod_label)

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor("#0F0F13")
ax.set_facecolor("#0F0F13")

color_map = {"Sentiment": SENT_COLOR, "LDA Topics": LDA_COLOR, "Embeddings": EMBED_COLOR}
for submod, grp in merged.groupby("submod"):
    ax.scatter(grp["abs_r_kaggle"], grp["abs_r_google"],
               color=color_map[submod], alpha=0.65, s=25, label=submod, zorder=3)

# Reference line y=x
lim_max = max(merged["abs_r_kaggle"].max(), merged["abs_r_google"].max()) * 1.05
ax.plot([0, lim_max], [0, lim_max], color="#666", linestyle="--",
        linewidth=1, label="y = x (equal effect)", zorder=2)

# Shade regions
ax.fill_between([0, lim_max], [0, 0], [0, lim_max], alpha=0.04,
                color=KAGGLE_COLOR, label="Kaggle > Google")
ax.fill_between([0, lim_max], [0, lim_max], [lim_max, lim_max],
                alpha=0.04, color=GOOGLE_COLOR, label="Google > Kaggle")

ax.set_xlabel("|Pearson r| — kaggle_ai", color="white", fontsize=12)
ax.set_ylabel("|Pearson r| — google_real", color="white", fontsize=12)
ax.set_title("Effect Size per Text Feature: Kaggle vs Google\n"
             "Below line = Kaggle has larger effect size",
             fontsize=12, color="white", pad=12)
ax.tick_params(colors="white")
ax.legend(fontsize=9, framealpha=0.2)
for spine in ax.spines.values(): spine.set_edgecolor("#333")
ax.grid(alpha=0.1)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "plot4_effect_size_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved plot4")


## 13. Save Feature-Level Detail

In [ ]:
# Merge all results into one detail file
detail_k = bonf_kaggle[["feature","p_value","pearson_r","bonferroni_rejected","fdr_rejected"]].copy()
detail_k = detail_k.merge(boot_kaggle[["feature","bootstrap_rejected"]], on="feature", how="left")
detail_k = detail_k.merge(wf_kaggle[["feature","wf_rejected","mean_gen_ratio"]], on="feature", how="left")
detail_k["source"] = "kaggle_ai"

detail_g = bonf_google[["feature","p_value","pearson_r","bonferroni_rejected","fdr_rejected"]].copy()
detail_g = detail_g.merge(boot_google[["feature","bootstrap_rejected"]], on="feature", how="left")
detail_g = detail_g.merge(wf_google[["feature","wf_rejected","mean_gen_ratio"]], on="feature", how="left")
detail_g["source"] = "google_real"

detail = pd.concat([detail_k, detail_g], ignore_index=True)
detail["submod"] = detail["feature"].map(submod_label)
detail.to_csv(os.path.join(OUTDIR, "source_split_feature_detail.csv"), index=False)
print(f"Saved {len(detail)} feature rows to source_split_feature_detail.csv")
print(f"All outputs in: {OUTDIR}")


## 14. Interpretation Guide

| Plot | What to look for |
|---|---|
| Plot 1 (grouped bars) | Δ consistently positive across all 4 methods = discrimination exists |
| Plot 2 (heatmap) | Red cells = that sub-modality/method pair discriminates; blue = reverse |
| Plot 3 (p-values) | Kaggle uniform + Google left-skewed = google has more genuine signal |
| Plot 4 (scatter) | Most points below y=x = kaggle features have larger raw effect sizes |

**If avg Δ > 0.05**: Methods do partially transfer to text — finding supports RQ1.

**If avg Δ ≈ 0**: Detection methods cannot distinguish AI-generated from real text features —
this is the null result and is itself a contribution (boundary condition for method transfer).
